In [1]:
import numpy as np
import scipy
import pandas as pd
import gensim
from gensim.models import Word2Vec

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Pandas:", pd.__version__)
print("Gensim:", gensim.__version__)
print("All imports successful!")

NumPy: 1.26.4
SciPy: 1.13.1
Pandas: 2.2.3
Gensim: 4.3.3
All imports successful!


In [2]:
import matplotlib.pyplot as plt
import re
import nltk
import warnings
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec


warnings.filterwarnings("ignore")
%matplotlib inline

In [3]:
for resource in ["tokenizers/punkt", "tokenizers/punkt_tab", "corpora/stopwords"]:
    try:
        nltk.data.find(resource)
        print(resource, "available")
    except LookupError:
        print(resource, "missing. Download it before going offline.")

tokenizers/punkt available
tokenizers/punkt_tab available
corpora/stopwords available


In [4]:
sentences = """Machine learning helps computers learn from data.
Students use Python to build intelligent applications.
Neural networks can recognize patterns in images and text.
Deep learning models require training data and computing power.
Natural language processing allows computers to understand human language.
Word embeddings represent words as numerical vectors.
Artificial intelligence is transforming healthcare, education, and finance.
Researchers continuously improve machine learning algorithms."""

print(sentences)

Machine learning helps computers learn from data.
Students use Python to build intelligent applications.
Neural networks can recognize patterns in images and text.
Deep learning models require training data and computing power.
Natural language processing allows computers to understand human language.
Word embeddings represent words as numerical vectors.
Artificial intelligence is transforming healthcare, education, and finance.
Researchers continuously improve machine learning algorithms.


In [5]:
sentences = re.sub(r"[^A-Za-z]+", " ", sentences)
sentences = re.sub(r"\b\w\b", " ", sentences).strip()
print(sentences)

Machine learning helps computers learn from data Students use Python to build intelligent applications Neural networks can recognize patterns in images and text Deep learning models require training data and computing power Natural language processing allows computers to understand human language Word embeddings represent words as numerical vectors Artificial intelligence is transforming healthcare education and finance Researchers continuously improve machine learning algorithms


In [6]:
sentences = sentences.lower()
all_sent = nltk.sent_tokenize(sentences)
all_words = [nltk.word_tokenize(s) for s in all_sent]
print(all_words)

[['machine', 'learning', 'helps', 'computers', 'learn', 'from', 'data', 'students', 'use', 'python', 'to', 'build', 'intelligent', 'applications', 'neural', 'networks', 'can', 'recognize', 'patterns', 'in', 'images', 'and', 'text', 'deep', 'learning', 'models', 'require', 'training', 'data', 'and', 'computing', 'power', 'natural', 'language', 'processing', 'allows', 'computers', 'to', 'understand', 'human', 'language', 'word', 'embeddings', 'represent', 'words', 'as', 'numerical', 'vectors', 'artificial', 'intelligence', 'is', 'transforming', 'healthcare', 'education', 'and', 'finance', 'researchers', 'continuously', 'improve', 'machine', 'learning', 'algorithms']]


In [7]:
stop_words = set(stopwords.words("english"))
data = [[w for w in sentence if w not in stop_words] for sentence in all_words]
data1 = data[0]
print(data)

[['machine', 'learning', 'helps', 'computers', 'learn', 'data', 'students', 'use', 'python', 'build', 'intelligent', 'applications', 'neural', 'networks', 'recognize', 'patterns', 'images', 'text', 'deep', 'learning', 'models', 'require', 'training', 'data', 'computing', 'power', 'natural', 'language', 'processing', 'allows', 'computers', 'understand', 'human', 'language', 'word', 'embeddings', 'represent', 'words', 'numerical', 'vectors', 'artificial', 'intelligence', 'transforming', 'healthcare', 'education', 'finance', 'researchers', 'continuously', 'improve', 'machine', 'learning', 'algorithms']]


In [8]:
model1 = Word2Vec(
    sentences=data,
    min_count=1,
    vector_size=52,
    window=5,
    workers=1,
    sg=0,
    seed=42,
    epochs=100
)
print("Word2Vec model trained.")

Word2Vec model trained.


In [9]:
vocabulary = model1.wv.index_to_key
print(vocabulary)

['learning', 'machine', 'computers', 'language', 'data', 'neural', 'models', 'deep', 'text', 'images', 'patterns', 'recognize', 'networks', 'intelligent', 'applications', 'training', 'build', 'python', 'use', 'students', 'learn', 'helps', 'require', 'algorithms', 'improve', 'power', 'continuously', 'researchers', 'finance', 'education', 'healthcare', 'transforming', 'intelligence', 'artificial', 'vectors', 'numerical', 'words', 'represent', 'embeddings', 'word', 'human', 'understand', 'allows', 'processing', 'natural', 'computing']


In [10]:
wrd = 'machine'

v1 = model1.wv[wrd]
similar_words = model1.wv.most_similar(wrd)

for x in similar_words:
    print(x)

('understand', 0.35685068368911743)
('allows', 0.3444419503211975)
('represent', 0.2987847924232483)
('intelligent', 0.27325019240379333)
('patterns', 0.25695106387138367)
('data', 0.25187328457832336)
('computing', 0.20459283888339996)
('word', 0.1966744065284729)
('continuously', 0.192421093583107)
('healthcare', 0.18782345950603485)


In [11]:
dat = []

for i in range(2, len(data1) - 2):
    context = [data1[i-2], data1[i-1], data1[i+1], data1[i+2]]
    target = data1[i]
    dat.append((context, target))

print(dat[:10])

[(['machine', 'learning', 'computers', 'learn'], 'helps'), (['learning', 'helps', 'learn', 'data'], 'computers'), (['helps', 'computers', 'data', 'students'], 'learn'), (['computers', 'learn', 'students', 'use'], 'data'), (['learn', 'data', 'use', 'python'], 'students'), (['data', 'students', 'python', 'build'], 'use'), (['students', 'use', 'build', 'intelligent'], 'python'), (['use', 'python', 'intelligent', 'applications'], 'build'), (['python', 'build', 'applications', 'neural'], 'intelligent'), (['build', 'intelligent', 'neural', 'networks'], 'applications')]


In [12]:
i = min(1, len(dat) - 1)

if dat:
    print("Context:", dat[i][0])
    print("Target:", dat[i][1])
    print("Predictions:", model1.predict_output_word(dat[i][0]))
else:
    print("Not enough words to create context examples.")

Context: ['learning', 'helps', 'learn', 'data']
Target: computers
Predictions: [('helps', 0.021744972), ('natural', 0.02174245), ('patterns', 0.021742286), ('learn', 0.021742225), ('neural', 0.021741955), ('students', 0.021741902), ('intelligence', 0.021741655), ('human', 0.021741532), ('healthcare', 0.021741383), ('networks', 0.021741217)]
